# 第三題：凍結後的事後分析

這不是凍結系統的一部分。計畫在執行前寫好：`evaluation/posthoc-plan.md`（sha256 `db5d29720d73394010122d6d2523862ac15e3eef727c261bca0124b8925ae872`）。結果不論好壞，都不用來調整 prompt、知識卡、模型或微調權重，也不改寫既有判定。

- **實驗 A**：Qwen3-8B 以 16 位元載入兩張 T4、不載微調權重，用凍結的 prompt、知識卡、greedy 解碼與重寫規則審同樣 8 篇測試文章，和既有的 4-bit 無微調輸出比較，回答「YMYL 標太多是不是 4-bit 量化造成」。
- **實驗 B**：與訓練相同的 4-bit 載入，在 12 筆訓練資料的監督答案上計算載入與不載入微調權重時的交叉熵，回答「微調有沒有讓模型更接近答案」。不讀測試資料。

程式從凍結 bundle 匯入、不修改；只有模型載入方式（實驗 A）與 loss 計算（實驗 B）是新寫的。

In [ ]:
import os,sys,json,hashlib,shutil,subprocess
from pathlib import Path
PLAN_SHA256='db5d29720d73394010122d6d2523862ac15e3eef727c261bca0124b8925ae872'
LOCK_SHA256='8ab9b7ae24fe5907965e6dc7b7e226bdcd7f74a845ccc2bfee995f2fc627d0cf'
HOLDOUT_SHA256='818b491f19541c7703cc311afd498fe1169c223a1c03fb4902e275b46b7a9106'
INPUT=Path('/kaggle/input');WORK=Path('/kaggle/working');ROOT=WORK/'work03';POST=WORK/'posthoc'
def find(name):return sorted(Path(d)/name for d,_,fs in os.walk(INPUT,followlinks=True) if name in fs)
locks=[p for p in find('frozen.json') if p.parent.name=='outputs' and p.parent.parent.name=='work03']
assert len(locks)==1,locks
assert hashlib.sha256(locks[0].read_bytes()).hexdigest()==LOCK_SHA256,'Mounted train bundle is not the frozen lock'
shutil.copytree(locks[0].parent.parent,ROOT,ignore=shutil.ignore_patterns('__pycache__','trainer'))
(WORK/'posthoc.py').write_text('"""Post-hoc analysis after the freeze (plan: evaluation/posthoc-plan.md). Not part of the frozen system.\n\nStages (each in its own process):\n  fp16  Experiment A: Qwen3-8B in 16-bit across two T4s, no adapter, on the 8 test articles. Prompt, knowledge,\n        messages(), generate() (greedy), audit_one() (one repair) and the output contract are imported unchanged\n        from the frozen bundle; only the model loader differs from the frozen 4-bit loader.\n  loss  Experiment B: answer-token cross-entropy of the 12 supervised targets, 4-bit base with and without the\n        frozen adapter, eval mode, built with the frozen examples(). Test articles are not read.\nWORK03_ROOT points at the copied frozen bundle; outputs go to POSTHOC_OUT, outside the bundle."""\nimport json\nimport os\nimport sys\nimport time\nfrom pathlib import Path\n\nROOT = Path(os.environ[\'WORK03_ROOT\'])\nOUT = Path(os.environ[\'POSTHOC_OUT\'])\nsys.path.insert(0, str(ROOT / \'scripts\'))\nimport workflow as wf  # noqa: E402  frozen bundle code, imported unchanged\n\n\ndef versions():\n    import importlib.metadata as meta\n    import torch\n    return dict(torch=torch.__version__, **{p: meta.version(p) for p in\n                                          [\'transformers\', \'peft\', \'bitsandbytes\', \'accelerate\', \'datasets\']})\n\n\ndef fp16():\n    import torch\n    from transformers import AutoTokenizer, AutoModelForCausalLM\n    c = wf.config()\n    assert torch.cuda.device_count() == 2, \'Experiment A needs both T4s for 16-bit weights\'\n    tok = AutoTokenizer.from_pretrained(c[\'model_id\'], revision=c[\'model_revision\'])\n    tok.pad_token = tok.eos_token\n    tok.padding_side = \'right\'\n    model = AutoModelForCausalLM.from_pretrained(c[\'model_id\'], revision=c[\'model_revision\'], torch_dtype=torch.float16,\n                                                 device_map=\'auto\', max_memory={0: \'9GiB\', 1: \'12GiB\'})\n    for i in range(2):\n        torch.cuda.reset_peak_memory_stats(i)\n    env = dict(started_utc=time.strftime(\'%Y-%m-%dT%H:%M:%SZ\', time.gmtime()), weights=\'float16, no adapter\',\n               model_id=c[\'model_id\'], revision=c[\'model_revision\'], parameters=model.num_parameters(),\n               footprint_bytes=model.get_memory_footprint(),\n               devices=sorted(set(str(v) for v in model.hf_device_map.values())),\n               gpus=[torch.cuda.get_device_name(i) for i in range(2)], versions=versions(),\n               holdout_sha256=wf.sha(ROOT / \'data/holdout8.csv\'), lock_sha256=wf.sha(ROOT / \'outputs/frozen.json\'))\n    wf.write(OUT / \'fp16_env.json\', env)\n    started = time.monotonic()\n    wf.batch(wf.data(\'holdout8\'), lambda msg: wf.generate(tok, model, msg), OUT / \'holdout_fp16.jsonl\')\n    env.update(seconds_total=time.monotonic() - started,\n               peak_gpu1_bytes_all_items=torch.cuda.max_memory_allocated(1),\n               note=\'peak_gpu_bytes inside each record is device 0 only (frozen generate()); device 1 peak is above\')\n    wf.write(OUT / \'fp16_env.json\', env)\n    print(json.dumps(env, ensure_ascii=False, indent=1))\n\n\ndef loss():\n    import torch\n    tok, model = wf.load(adapter=True)\n    model.eval()\n    c = wf.config()\n    rows, labels = wf.data(\'train12\'), wf.targets()\n    ds = wf.examples(tok, rows, labels, c[\'max_length\'])\n\n    def answer_ce(ex):\n        ids = torch.tensor([ex[\'input_ids\']], device=\'cuda:0\')\n        lab = torch.tensor([ex[\'labels\']], device=\'cuda:0\')\n        j0 = max(1, int((lab != -100).nonzero(as_tuple=True)[1].min()))\n        with torch.inference_mode():\n            out = model(input_ids=ids, attention_mask=torch.ones_like(ids), logits_to_keep=lab.shape[1] - j0 + 1)\n        logits = out.logits[:, :-1, :].float()\n        tgt = lab[:, j0:]\n        s = torch.nn.functional.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1),\n                                              ignore_index=-100, reduction=\'sum\')\n        return float(s), int((tgt != -100).sum())\n\n    items = []\n    for row, ex in zip(rows, ds):\n        a_sum, n = answer_ce(ex)\n        with model.disable_adapter():\n            b_sum, n2 = answer_ce(ex)\n        assert n == n2\n        items.append(dict(ID=row[\'ID\'], answer_tokens=n, ce_adapter=a_sum / n, ce_base=b_sum / n,\n                          ce_sum_adapter=a_sum, ce_sum_base=b_sum))\n    tokens = sum(x[\'answer_tokens\'] for x in items)\n    ce_a = sum(x[\'ce_sum_adapter\'] for x in items) / tokens\n    ce_b = sum(x[\'ce_sum_base\'] for x in items) / tokens\n    result = dict(created_utc=time.strftime(\'%Y-%m-%dT%H:%M:%SZ\', time.gmtime()),\n                  setup=\'4-bit NF4 base as in training, eval mode, answer tokens only, frozen examples(); \'\n                        \'base = same model with the adapter disabled\',\n                  answer_tokens=tokens, ce_base=ce_b, ce_adapter=ce_a, relative_reduction=(ce_b - ce_a) / ce_b,\n                  items=items, versions=versions())\n    wf.write(OUT / \'train_target_loss.json\', result)\n    print(json.dumps({k: v for k, v in result.items() if k != \'items\'}, ensure_ascii=False, indent=1))\n\n\nif __name__ == \'__main__\':\n    OUT.mkdir(parents=True, exist_ok=True)\n    {\'fp16\': fp16, \'loss\': loss}[sys.argv[1]]()\n',encoding='utf-8')
subprocess.run([sys.executable,'-m','pip','install',*['transformers==4.57.6', 'peft==0.20.0', 'accelerate==1.14.0', 'bitsandbytes==0.50.2', 'datasets==4.8.5', 'jsonschema>=4,<5', 'pandas>=2,<3']],check=True)
env={**os.environ,'WORK03_ROOT':str(ROOT),'POSTHOC_OUT':str(POST),'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}
# Frozen check: every locked file (prompt, knowledge, config, data, scripts, adapter) must match the lock.
subprocess.run([sys.executable,str(ROOT/'scripts/workflow.py'),'check-freeze'],check=True,env=env)

## 實驗 A：16 位元、不載微調，審 8 篇測試文章
核對測試資料雜湊後才讀取；兩張 T4 分攤權重。

In [ ]:
holdouts=find('holdout8.csv')
assert len(holdouts)==1,holdouts
assert hashlib.sha256(holdouts[0].read_bytes()).hexdigest()==HOLDOUT_SHA256,'holdout8.csv bytes changed'
shutil.copyfile(holdouts[0],ROOT/'data/holdout8.csv')
subprocess.run([sys.executable,str(WORK/'posthoc.py'),'fp16'],check=True,env={**env,'CUDA_VISIBLE_DEVICES':'0,1'})

## 實驗 B：微調前後在 12 筆監督答案上的交叉熵
不讀測試資料；同一個 4-bit 模型，分別啟用與停用微調權重。

In [ ]:
subprocess.run([sys.executable,str(WORK/'posthoc.py'),'loss'],check=True,env={**env,'CUDA_VISIBLE_DEVICES':'0'})

## 結果摘要
逐筆判讀與指標在本機依預先登錄的計畫計算。

In [ ]:
import pandas as pd
from IPython.display import display
rows=[json.loads(s) for s in (POST/'holdout_fp16.jsonl').read_text(encoding='utf-8').splitlines() if s.strip()]
display(pd.DataFrame([dict(ID=r['ID'],first_pass_valid=r['first_pass_valid'],final_valid=r['final_valid'],
  Score=(r['parsed'] or {}).get('Score'),codes=','.join(x['code'] for x in (r['parsed'] or {}).get('Reasons',[])),
  severities=','.join(x['severity'] for x in (r['parsed'] or {}).get('Reasons',[]))) for r in rows]))
loss=json.loads((POST/'train_target_loss.json').read_text(encoding='utf-8'))
print({k:loss[k] for k in ['answer_tokens','ce_base','ce_adapter','relative_reduction']})
display(pd.DataFrame(loss['items']))
(POST/'plan_sha256.txt').write_text(PLAN_SHA256+'\n',encoding='utf-8')
# Keep only the post-hoc outputs; the frozen bundle and the test CSV stay in their original kernels and dataset.
shutil.rmtree(ROOT)
print(sorted(p.name for p in POST.iterdir()))